In [3]:
# IPA Phonemizer: https://github.com/bootphon/phonemizer
import phonemizer
import torch
from nltk.tokenize import word_tokenize

from tts.tokenizer.letters import SYMBOL_DICTS, EOS, BOS


class TextCleaner:
    def __init__(self, dummy: None = None):
        self.word_index_dictionary = SYMBOL_DICTS
        self.index_word_dictionary = {idx: sym for sym, idx in self.word_index_dictionary.items()}
        print(len(SYMBOL_DICTS))

    def __call__(self, text: str):
        indexes = []
        for char in text:
            try:
                indexes.append(self.word_index_dictionary[char])
            except KeyError:
                # print(text)
                pass
        return indexes

    def decode(self, token_ids: torch.Tensor | list[int]) -> str:
        """
        Decode token ids back to raw IPA symbol sequence.
        BOS/EOS are preserved if they are included in token_ids.

        Args:
            token_ids: (T,) LongTensor or list[int]

        Returns:
            Raw IPA sequence string.
        """
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.detach().cpu().long().tolist()

        return "".join(self.index_word_dictionary.get(int(idx), "") for idx in token_ids)


class TextTokenizer:
    def __init__(self):
        self.cleaner = TextCleaner()
        self.phonemizer = (
            phonemizer.backend.EspeakBackend(  # pyright: ignore[reportAttributeAccessIssue]
                language="en-us",
                preserve_punctuation=True,
                with_stress=True,
            )
        )

    def __call__(
        self,
        text: str,
    ) -> torch.Tensor:
        ps = self.to_ipa(text)
        tokens = self.cleaner(ps)
        tokens.insert(0, SYMBOL_DICTS[BOS])
        tokens.append(SYMBOL_DICTS[EOS])
        tokens = torch.LongTensor(tokens).unsqueeze(0)
        return tokens

    def to_ipa(self, text: str) -> str:
        text = text.strip()
        text = text.replace('"', "")
        ps = self.phonemizer.phonemize([text])
        ps = word_tokenize(ps[0])
        ps = " ".join(ps)
        return ps

    def decode(self, token_ids: torch.Tensor) -> str | list[str]:
        """
        Decode LongTensor token ids to raw IPA sequence.

        Args:
            token_ids:
                (T,) or (1, T) or (B, T) LongTensor.

        Returns:
            str if input is 1D or batch size 1, otherwise list[str].
        """
        if token_ids.dim() == 1:
            return self.cleaner.decode(token_ids)

        if token_ids.dim() != 2:
            raise ValueError(
                f"decode expects token_ids with shape (T,) or (B, T), "
                + f"but got {tuple(token_ids.shape)}."
            )

        decoded = [self.cleaner.decode(row) for row in token_ids]

        if len(decoded) == 1:
            return decoded[0]

        return decoded

    @property
    def n_vocab(self):
        return max(self.cleaner.word_index_dictionary.values()) + 1


In [4]:
len(SYMBOL_DICTS)

179

In [6]:
tokenizer = TextTokenizer()

tokens = tokenizer("hello world")      # (1, T)
raw_ipa = tokenizer.decode(tokens)     # BOS/EOS 포함 string
print(tokens)
print(raw_ipa)

179
tensor([[  2,  52,  85,  56, 159,  59, 138,  18,  67, 159,  89, 161,  56,  48,
           1]])
^həlˈoʊ wˈɜːld~


In [2]:
import torch

sd = torch.load("/home/blue2959/monotonic_tts/checkpoints/stage2-all-new/base/vctk/ckpt_step_505000_vctk.pth")
print(sd.keys())


dict_keys(['model', 'optimizer', 'scheduler', 'step', 'epoch'])


In [3]:
sd["model"].keys()

odict_keys(['syn_backbone.text_embedder.out.weight', 'syn_backbone.crf_aligner.unary_predictor.evidence_scale', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.0.weight', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.0.bias', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.1.weight', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.1.bias', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.3.weight', 'syn_backbone.crf_aligner.unary_predictor.spec_proj.3.bias', 'syn_backbone.crf_aligner.unary_predictor.text_proj.0.weight', 'syn_backbone.crf_aligner.unary_predictor.text_proj.0.bias', 'syn_backbone.crf_aligner.unary_predictor.text_proj.1.weight', 'syn_backbone.crf_aligner.unary_predictor.text_proj.1.bias', 'syn_backbone.crf_aligner.unary_predictor.text_proj.3.weight', 'syn_backbone.crf_aligner.unary_predictor.text_proj.3.bias', 'syn_backbone.crf_aligner.unary_predictor.cond_proj.weight', 'syn_backbone.crf_aligner.unary_predictor.cond_proj.bias', 'syn_backbone.crf_aligne